# Lesson 9: Urban Growth with Machine Learning

**Planning question: How has the city expanded?**

Bengaluru has grown very fast, especially to the east along the IT corridors. In this notebook you will:

1. Train a **machine learning** model to recognise land cover (built-up, vegetation, water, bare) from satellite images
2. Use it to map land cover in **2019** and **2024**
3. Measure where, and how much, new built-up area has appeared

The study area is east Bengaluru: Whitefield, Marathahalli, Varthur and Bellandur.

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q pystac-client planetary-computer odc-stac rioxarray

In [ ]:
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from shapely.geometry import shape, box
from shapely import union_all
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
BBOX = [77.64, 12.90, 77.80, 13.02]   # east Bengaluru
UTM_CRS = "EPSG:32643"
RESOLUTION = 20                        # metres
YEARS = [2019, 2021, 2024]             # 2021 is used for training

CLASS_NAMES = {1: "Built-up", 2: "Vegetation", 3: "Water", 4: "Bare"}
CLASS_COLORS = ListedColormap(["#d7301f", "#41ab5d", "#2171b5", "#fdd49e"])

## Step 1: A function to load a clear Sentinel-2 image for any year

This combines the steps from Lessons 6 and 7 into one function: search, pick the clearest date, load, convert to reflectance,
mask clouds and add the NDVI and NDBI indices. We use six bands, because more bands help the model tell surfaces apart.

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]


def load_sentinel2(year):
    items = catalog.search(collections=["sentinel-2-l2a"], bbox=BBOX,
                           datetime=f"{year}-01-01/{year}-03-31",
                           query={"eo:cloud_cover": {"lt": 10}}).item_collection()

    # pick the least cloudy date that covers the whole box
    by_date = {}
    for item in items:
        by_date.setdefault(item.datetime.date(), []).append(item)
    options = [(np.mean([i.properties["eo:cloud_cover"] for i in its]), d, its)
               for d, its in by_date.items()
               if union_all([shape(i.geometry) for i in its]).contains(box(*BBOX))]
    cloud, date, best = min(options, key=lambda option: option[0])
    print(f"{year}: using {date} ({cloud:.1f}% cloud)")

    ds = odc.stac.load(best, bands=BANDS + ["SCL"], bbox=BBOX, crs=UTM_CRS,
                       resolution=RESOLUTION, groupby="solar_day").isel(time=0)

    offset = 1000 if float(best[0].properties["s2:processing_baseline"]) >= 4 else 0
    clear = ds["SCL"].isin([4, 5, 6])
    out = xr.Dataset({b: ((ds[b].astype("float32") - offset) / 10000).where(clear) for b in BANDS})
    out["NDVI"] = (out["B08"] - out["B04"]) / (out["B08"] + out["B04"])
    out["NDBI"] = (out["B11"] - out["B08"]) / (out["B11"] + out["B08"])
    return out


images = {year: load_sentinel2(year) for year in YEARS}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, year in zip(axes, [2019, 2024]):
    images[year][["B04", "B03", "B02"]].to_array("band").plot.imshow(ax=ax, rgb="band", robust=True)
    ax.set_title(f"True colour {year}")
    ax.set_axis_off()
plt.show()

## Step 2: Training labels from ESA WorldCover

A **supervised** model learns from examples where we already know the answer.
**ESA WorldCover** is a global land cover map for 2021 at 10 m. We use it as the "answer key" for our 2021 image.

We load it on exactly the same grid as our images, then group its 11 classes into our 4.

In [ ]:
wc_items = catalog.search(collections=["esa-worldcover"], bbox=BBOX,
                          datetime="2021-01-01/2021-12-31").item_collection()
worldcover = odc.stac.load(wc_items, bands=["map"], geobox=images[2021].odc.geobox,
                           resampling="nearest", groupby="solar_day")["map"].isel(time=0)

labels = xr.zeros_like(worldcover, dtype="int8")
labels = labels.where(worldcover != 50, 1)                                  # built-up
labels = labels.where(~worldcover.isin([10, 20, 30, 40, 90, 95, 100]), 2)  # trees, shrubs, grass, crops, wetland
labels = labels.where(worldcover != 80, 3)                                  # water
labels = labels.where(worldcover != 60, 4)                                  # bare

labels.where(labels > 0).plot(cmap=CLASS_COLORS, vmin=0.5, vmax=4.5, size=7, add_colorbar=False)
plt.title("Training labels from ESA WorldCover 2021\nred = built-up, green = vegetation, blue = water, sand = bare")
plt.show()

## Step 3: Turn the image into a table

Machine learning works on tables. Each **pixel** becomes a **row**, and each band or index becomes a **column** (a feature).

In [ ]:
FEATURES = BANDS + ["NDVI", "NDBI"]


def to_table(ds):
    return ds[FEATURES].to_dataframe(dim_order=["y", "x"])[FEATURES]


table_2021 = to_table(images[2021])
table_2021["label"] = labels.values.ravel()
table_2021 = table_2021.dropna()
table_2021 = table_2021[table_2021["label"] > 0]
table_2021["label"].map(CLASS_NAMES).value_counts()

## Step 4: Train the Random Forest

Some classes, such as water, are rare. To keep things fair, we take up to 3,000 random pixels from each class.
We keep 30 percent aside as a **test set** to check the model on pixels it has never seen.

A **Random Forest** is a group of decision trees. Each tree learns simple rules, such as "if NDVI is above 0.4, it is vegetation",
and the forest takes a vote.

In [ ]:
sample = table_2021.groupby("label").sample(n=3000, replace=True, random_state=42).drop_duplicates()
X_train, X_test, y_train, y_test = train_test_split(
    sample[FEATURES], sample["label"], test_size=0.3, random_state=42, stratify=sample["label"]
)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

## Step 5: How accurate is it?

- **Precision**: when the model says "built-up", how often is it right?
- **Recall**: of all real built-up pixels, how many did the model find?

In [ ]:
predicted = model.predict(X_test)
print(classification_report(y_test, predicted, target_names=list(CLASS_NAMES.values())))
ConfusionMatrixDisplay.from_predictions(y_test, predicted, display_labels=list(CLASS_NAMES.values()), cmap="Blues")
plt.show()

Which features did the model rely on most?

In [ ]:
pd.Series(model.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(6, 4), color="teal")
plt.title("Feature importance")
plt.show()

## Step 6: Classify 2019 and 2024

Now we apply the trained model to every pixel of the 2019 and 2024 images.

In [ ]:
def classify(ds):
    table = to_table(ds)
    valid = table.dropna()
    result = pd.Series(model.predict(valid), index=valid.index).reindex(table.index)
    grid = result.values.reshape(ds.sizes["y"], ds.sizes["x"])
    return xr.DataArray(grid, coords={"y": ds.y, "x": ds.x}, dims=("y", "x"))


landcover = {year: classify(images[year]) for year in [2019, 2024]}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, year in zip(axes, [2019, 2024]):
    landcover[year].plot(ax=ax, cmap=CLASS_COLORS, vmin=0.5, vmax=4.5, add_colorbar=False)
    ax.set_title(f"Land cover {year}")
    ax.set_axis_off()
plt.show()

## Step 7: How much has changed?

Each pixel is 20 m by 20 m, which is 400 square metres. Counting the pixels gives the area.

In [ ]:
pixel_km2 = RESOLUTION ** 2 / 1_000_000
areas = pd.DataFrame({
    year: landcover[year].to_series().value_counts().sort_index() * pixel_km2
    for year in [2019, 2024]
}).rename(index=CLASS_NAMES).round(1)
areas["change_km2"] = areas[2024] - areas[2019]
areas

## Step 8: Map the new built-up areas

New built-up pixels are those classed as vegetation or bare in 2019 that became built-up in 2024.

In [ ]:
before, after = landcover[2019], landcover[2024]
new_built = (after == 1) & before.isin([2, 4])

fig, ax = plt.subplots(figsize=(10, 8))
images[2024][["B04", "B03", "B02"]].to_array("band").plot.imshow(ax=ax, rgb="band", robust=True)
new_built.where(new_built).plot(ax=ax, cmap=ListedColormap(["magenta"]), add_colorbar=False)
ax.set_title(f"New built-up area 2019 to 2024 (magenta): {float(new_built.sum()) * pixel_km2:.1f} square km")
ax.set_axis_off()
plt.show()

## Step 9: Save

In [ ]:
for year in [2019, 2024]:
    landcover[year].rio.write_crs(UTM_CRS).astype("float32").rio.to_raster(f"landcover_{year}.tif")
new_built.astype("uint8").rio.write_crs(UTM_CRS).rio.to_raster("new_built_2019_2024.tif")
print("Saved")

## What this means for planning

- Classified images show where growth is really happening, which may differ from where the master plan expected it.
- New built-up area that eats into lakes, wetlands and farmland is an early warning for flooding and the loss of green belts.
- The same method can track growth around a new metro line, ring road or airport.

**Limitations:** the model learned from 2021 and is applied to other years, so different light and season can cause errors.
Small changes of a few pixels may be noise, so check them against the true colour images.
For official figures, collect your own ground truth points and assess accuracy for each year.

## Try it yourself

1. Change `BBOX` to north Bengaluru near the airport: `[77.58, 13.08, 77.74, 13.20]`.
2. Remove `"NDVI"` and `"NDBI"` from `FEATURES` and train again. Does accuracy drop?
3. Change `n_estimators` from 100 to 20. How much faster is it, and is it less accurate?